# Exploring the data (base network)

In [1]:
!pip install folium geopandas pandas pypsa mapclassify matplotlib numpy shapely


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import geopandas as gpd
import numpy as np
import pypsa

In [7]:
resources = "resources/networks/"

!snakemake resources/networks/base.nc --configfile config/config_ES.yaml --cores 4

WorkflowError in file "/workspaces/SEE/Mapas/pypsa-spain/rules/retrieve.smk", line 87:
No storage provider found for query https://gisco-services.ec.europa.eu/distribution/v2/nuts/download/ref-nuts-2013-03m.geojson.zip. Either install the required storage plugin or check your query. Also consider to explicitly specify the storage provider to get a more informative error message.


Import the base network

In [8]:
n = pypsa.Network(resources + "base.nc")

FileNotFoundError: [Errno 2] No such file or directory: '/workspaces/SEE/Mapas/pypsa-spain/resources/networks/base.nc'

PyPSA calculates the line parameters just before the model is solved. If we want to see the underlying impedance $r$, reactance $x$, and susceptance $b$, we can trigger the calculation, manually.

In [ ]:
n.calculate_dependent_values()
n.lines["i_nom"] = (
    (n.lines.s_nom / n.lines.v_nom / n.lines.num_parallel).div(np.sqrt(3)).round(3)
)  # kA

PyPSA maps each line to a library of built-in standard line types to obtain per line type, per km values.

In [ ]:
print(n.line_types.head())
print("\nAC line types in the base network:")
print(f"AC Line types: {sorted(n.lines.type.unique())}")

We import a helper function to visualise the detailed network, interactively. Note that you do not need to understand the function `create_geometries()`. It is only needed to create shapely geometries from the network data.

In [ ]:
def create_geometries(network):
    """
    Create GeoDataFrames for different network components with specified coordinate reference system (CRS).

    Parameters
    ----------
        network (PyPSA Network): The network object containing buses, lines, links, converters, and transformers data.
        is_converter (bool): Boolean that specifies if link element is a converter.
        crs (str, optional): Coordinate reference system to be used for the GeoDataFrames. Defaults to GEO_CRS.

    Returns
    -------
    tuple: A tuple containing the following GeoDataFrames:
        - buses (GeoDataFrame): GeoDataFrame containing bus data with geometries.
        - lines (GeoDataFrame): GeoDataFrame containing line data with geometries.
        - links (GeoDataFrame): GeoDataFrame containing link data with geometries.
        - converters (GeoDataFrame): GeoDataFrame containing converter data with geometries.
        - transformers (GeoDataFrame): GeoDataFrame containing transformer data with geometries.
    """
    import geopandas as gpd
    from shapely.wkt import loads

    crs = network.crs

    network.buses["dc"] = network.buses["carrier"].map({"DC": "t", "AC": "f"})
    buses = network.buses.reset_index()[
        [
            "Bus",
            "v_nom",
            "dc",
            "symbol",
            "under_construction",
            "tags",
            "geometry",
        ]
    ]
    buses["geometry"] = buses.geometry.apply(lambda x: loads(x))
    buses = gpd.GeoDataFrame(buses, geometry="geometry", crs=crs)

    lines = network.lines.reset_index()[
        [
            "Line",
            "bus0",
            "bus1",
            "v_nom",
            "i_nom",
            "num_parallel",
            "s_nom",
            "r",
            "x",
            "b",
            "length",
            "underground",
            "under_construction",
            "type",
            "tags",
            "geometry",
        ]
    ]
    # Create shapely linestring from geometry column
    lines["geometry"] = lines.geometry.apply(lambda x: loads(x))
    lines = gpd.GeoDataFrame(lines, geometry="geometry", crs=crs)

    is_converter = network.links.index.str.startswith("conv")
    links = (
        network.links[~is_converter]
        .reset_index()
        .rename(columns={"voltage": "v_nom"})[
            [
                "Link",
                "bus0",
                "bus1",
                "v_nom",
                "p_nom",
                "length",
                "underground",
                "under_construction",
                "tags",
                "geometry",
            ]
        ]
    )
    links["geometry"] = links.geometry.apply(lambda x: loads(x))
    links = gpd.GeoDataFrame(links, geometry="geometry", crs=crs)

    converters = (
        network.links[is_converter]
        .reset_index()
        .rename(columns={"voltage": "v_nom"})[
            [
                "Link",
                "bus0",
                "bus1",
                "v_nom",
                "p_nom",
                "geometry",
            ]
        ]
    )
    converters["geometry"] = converters.geometry.apply(lambda x: loads(x))
    converters = gpd.GeoDataFrame(converters, geometry="geometry", crs=crs)

    transformers = network.transformers.reset_index()[
        [
            "Transformer",
            "bus0",
            "bus1",
            "voltage_bus0",
            "voltage_bus1",
            "s_nom",
            "geometry",
        ]
    ]
    transformers["geometry"] = transformers.geometry.apply(lambda x: loads(x))
    transformers = gpd.GeoDataFrame(transformers, geometry="geometry", crs=crs)

    return buses, lines, links, converters, transformers

Apply the function to the network:

In [ ]:
buses, lines, links, converters, transformers = create_geometries(n)

# Interactive map

Using the PyPSA base network, let's create an **interactive map**. To help visualise the underlying input data, we also import the cleaned substations.

In [ ]:
stations_polygon = gpd.read_file(resources + "stations_polygon.geojson")
buses_polygon = gpd.read_file(resources + "buses_polygon.geojson")

Stacking everything together on a single folium map

In [ ]:
map = None
b_popup = True

map = stations_polygon.explore(color="yellow", popup=b_popup)
map = buses_polygon.query("dc == False").explore(color="red", popup=b_popup, m=map)
map = buses_polygon.query("dc == True").explore(color="purple", popup=b_popup, m=map)
map = lines.query("v_nom <= 230").explore(color="green", popup=b_popup, m=map)
map = lines.query("(v_nom > 230) & (v_nom <= 330)").explore(
    color="orange", popup=b_popup, m=map
)
map = lines.query("(v_nom > 330) & (v_nom <= 420)").explore(
    color="darkred", popup=b_popup, m=map
)
map = lines.query("v_nom > 420").explore(color="pink", popup=b_popup, m=map)
map = links.explore(color="purple", popup=b_popup, m=map)
map = converters.explore(color="black", popup=b_popup, m=map)
map = transformers.explore(color="pink", popup=b_popup, m=map)

In [ ]:
#map